# Broadcast to Tactical Radar Map

Convert football broadcast footage into a 2D tactical mini-map.
Detects players, assigns teams by jersey color, and projects onto a top-down pitch.

**Pipeline:** Video → YOLOv8 → ByteTrack → K-means team color → Homography → 2D Radar

In [ ]:
!pip install -q ultralytics supervision opencv-python-headless numpy matplotlib yt-dlp

import cv2, numpy as np, os
import matplotlib.pyplot as plt
from ultralytics import YOLO
import supervision as sv
from sklearn.cluster import KMeans

print('Setup complete')

## Step 1: Get a Football Clip

Either upload your own or download from YouTube.

In [ ]:
# Option A: Download a clip from YouTube (first 30 seconds)
!yt-dlp -f 'best[height<=720]' --download-sections '*0-30' -o '/content/match_clip.mp4' 'https://www.youtube.com/watch?v=YOUTUBE_ID_HERE' 2>/dev/null || true

# Option B: Upload your own
from google.colab import files
if not os.path.exists('/content/match_clip.mp4'):
    print('Upload a football broadcast clip:')
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        os.rename(fname, '/content/match_clip.mp4')

# Verify
cap = cv2.VideoCapture('/content/match_clip.mp4')
w, h, fps = int(cap.get(3)), int(cap.get(4)), cap.get(5)
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f'Video: {w}x{h} @ {fps:.0f}fps, {frames} frames ({frames/fps:.1f}s)')

## Step 2: Detect Players + Ball

In [ ]:
# YOLOv8 for person (class 0) and sports ball (class 32)
model = YOLO('yolov8n.pt')  # nano for speed; use yolov8m.pt for accuracy

# Process first frame to verify
cap = cv2.VideoCapture('/content/match_clip.mp4')
ret, frame = cap.read()
cap.release()

results = model(frame, classes=[0, 32], conf=0.3, verbose=False)
detections = sv.Detections.from_ultralytics(results[0])
print(f'Detected: {len(detections)} objects (persons + ball)')

# Annotate and show
annotator = sv.BoxAnnotator(thickness=2)
annotated = annotator.annotate(frame.copy(), detections)
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title(f'Detections: {len(detections)} objects'); plt.axis('off'); plt.show()

## Step 3: Team Classification (Jersey Color)

In [ ]:
def get_dominant_color(frame, bbox, n_colors=1):
    """Extract dominant jersey color from upper half of bounding box."""
    x1, y1, x2, y2 = map(int, bbox)
    # Upper half = torso (jersey), avoid shorts/grass
    h_half = (y2 - y1) // 2
    crop = frame[y1:y1+h_half, x1:x2]
    if crop.size == 0: return np.array([0, 0, 0])
    pixels = crop.reshape(-1, 3).astype(float)
    # Remove green (grass) pixels
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    mask = ~((hsv[:,:,0] > 30) & (hsv[:,:,0] < 85) & (hsv[:,:,1] > 40))
    pixels = crop[mask].reshape(-1, 3).astype(float)
    if len(pixels) < 10: return np.array([128, 128, 128])
    km = KMeans(n_clusters=n_colors, n_init=3, random_state=0).fit(pixels)
    return km.cluster_centers_[0]

# Classify each detection into teams
person_mask = detections.class_id == 0
person_dets = detections[person_mask]

colors = [get_dominant_color(frame, bbox) for bbox in person_dets.xyxy]
colors = np.array(colors)

# Cluster into 3 groups: Team A, Team B, Referee
if len(colors) >= 3:
    team_km = KMeans(n_clusters=3, n_init=5, random_state=42).fit(colors)
    team_labels = team_km.labels_
    centers = team_km.cluster_centers_
    print(f'Team centers (BGR): {centers.astype(int).tolist()}')
    
    # Assign: largest two clusters = teams, smallest = referees
    counts = [np.sum(team_labels == i) for i in range(3)]
    sorted_idx = np.argsort(counts)[::-1]
    team_a_idx, team_b_idx, ref_idx = sorted_idx[0], sorted_idx[1], sorted_idx[2]
    print(f'Team A: {counts[team_a_idx]} players, Team B: {counts[team_b_idx]}, Ref: {counts[ref_idx]}')
else:
    team_labels = np.zeros(len(colors), dtype=int)
    print('Not enough detections for team classification')

## Step 4: Pitch Registration (Manual 4-Point Homography)

Select 4 known points on the pitch to compute the perspective transform.

In [ ]:
# FIFA standard pitch: 105m x 68m
PITCH_LENGTH = 105.0
PITCH_WIDTH = 68.0

# For automated homography, you'd use a pitch keypoint detection model.
# For this demo, we define 4 points manually from the broadcast frame.
# UPDATE THESE for your specific clip:

# Source points: pixel coordinates in the broadcast frame
# (click on 4 known points: e.g., penalty box corners)
# Format: [[x, y], ...]
pts_src = np.array([
    [200, 400],   # left penalty box near corner
    [600, 400],   # right penalty box near corner
    [500, 250],   # right penalty box far corner
    [250, 250],   # left penalty box far corner
], dtype=np.float32)

# Destination points: real-world coordinates in meters
# (corresponding points on a standard pitch)
# Penalty box: 16.5m from goal line, 40.32m wide (centered)
pts_dst = np.array([
    [0, (PITCH_WIDTH - 40.32) / 2],           # left near
    [0, (PITCH_WIDTH + 40.32) / 2],           # right near
    [16.5, (PITCH_WIDTH + 40.32) / 2],        # right far
    [16.5, (PITCH_WIDTH - 40.32) / 2],        # left far
], dtype=np.float32)

H, status = cv2.findHomography(pts_src, pts_dst)
print(f'Homography matrix computed: {status.sum()}/{len(status)} inliers')
print('NOTE: Update pts_src coordinates for your specific video clip!')

## Step 5: Render 2D Tactical Radar

In [ ]:
def draw_pitch(ax, length=105, width=68):
    """Draw a standard football pitch on matplotlib axis."""
    ax.set_xlim(-2, length + 2)
    ax.set_ylim(-2, width + 2)
    ax.set_aspect('equal')
    ax.set_facecolor('#2d8a3e')
    # Pitch outline
    ax.plot([0, length, length, 0, 0], [0, 0, width, width, 0], 'w-', lw=2)
    # Halfway line
    ax.plot([length/2, length/2], [0, width], 'w-', lw=1.5)
    # Center circle
    circle = plt.Circle((length/2, width/2), 9.15, fill=False, color='w', lw=1.5)
    ax.add_patch(circle)
    # Penalty boxes
    for x in [0, length - 16.5]:
        ax.plot([x, x+16.5, x+16.5, x], 
               [(width-40.32)/2, (width-40.32)/2, (width+40.32)/2, (width+40.32)/2], 'w-', lw=1.5)
    # Goal boxes
    for x in [0, length - 5.5]:
        ax.plot([x, x+5.5, x+5.5, x],
               [(width-18.32)/2, (width-18.32)/2, (width+18.32)/2, (width+18.32)/2], 'w-', lw=1)
    ax.set_xticks([]); ax.set_yticks([])

def project_to_pitch(detections, team_labels, H, frame):
    """Project player foot positions to pitch coordinates."""
    positions = []
    for i, bbox in enumerate(detections.xyxy):
        # Bottom center of bbox = foot position
        foot_x = (bbox[0] + bbox[2]) / 2
        foot_y = bbox[3]
        pt = np.array([[[foot_x, foot_y]]], dtype=np.float32)
        projected = cv2.perspectiveTransform(pt, H)
        px, py = projected[0][0]
        if 0 <= px <= PITCH_LENGTH and 0 <= py <= PITCH_WIDTH:
            positions.append((px, py, team_labels[i] if i < len(team_labels) else 0))
    return positions

# Project and render
positions = project_to_pitch(person_dets, team_labels, H, frame)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: annotated broadcast frame
ax1.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
ax1.set_title('Broadcast View'); ax1.axis('off')

# Right: 2D tactical radar
draw_pitch(ax2)
team_colors_map = {team_a_idx: 'blue', team_b_idx: 'red', ref_idx: 'yellow'} if len(colors) >= 3 else {0: 'white'}
for px, py, team in positions:
    color = team_colors_map.get(team, 'white')
    ax2.plot(px, py, 'o', color=color, markersize=10, markeredgecolor='white', markeredgewidth=1)
ax2.set_title('Tactical Radar Map')

plt.tight_layout()
plt.savefig('/content/tactical_radar_output.png', dpi=150)
plt.show()
print('Saved: /content/tactical_radar_output.png')

## Step 6: Process Full Video with Tracking

In [ ]:
# Process video with ByteTrack for persistent IDs
tracker = sv.ByteTrack()
box_annotator = sv.BoxAnnotator(thickness=1)
label_annotator = sv.LabelAnnotator(text_scale=0.4)

cap = cv2.VideoCapture('/content/match_clip.mp4')
w, h, fps = int(cap.get(3)), int(cap.get(4)), cap.get(5)
out = cv2.VideoWriter('/content/tactical_output.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (w*2, h))

frame_count = 0
MAX_FRAMES = int(fps * 15)  # Process 15 seconds

while cap.isOpened() and frame_count < MAX_FRAMES:
    ret, frame = cap.read()
    if not ret: break
    
    # Detect
    results = model(frame, classes=[0, 32], conf=0.3, verbose=False)
    dets = sv.Detections.from_ultralytics(results[0])
    dets = tracker.update_with_detections(dets)
    
    # Team colors (every 30 frames re-cluster)
    person_mask = dets.class_id == 0
    person_dets_frame = dets[person_mask]
    
    if len(person_dets_frame) >= 3 and frame_count % 30 == 0:
        colors_frame = np.array([get_dominant_color(frame, b) for b in person_dets_frame.xyxy])
        km_frame = KMeans(n_clusters=3, n_init=3, random_state=42).fit(colors_frame)
        labels_frame = km_frame.labels_
    elif 'labels_frame' not in dir() or len(person_dets_frame) != len(labels_frame):
        labels_frame = np.zeros(len(person_dets_frame), dtype=int)
    
    # Annotate broadcast frame
    annotated_frame = box_annotator.annotate(frame.copy(), dets)
    
    # Create radar
    radar = np.ones((h, w, 3), dtype=np.uint8) * 45  # dark green bg
    # Draw simple pitch lines on radar image
    margin = 30
    pw, ph = w - 2*margin, h - 2*margin
    cv2.rectangle(radar, (margin, margin), (margin+pw, margin+ph), (255,255,255), 2)
    cv2.line(radar, (margin+pw//2, margin), (margin+pw//2, margin+ph), (255,255,255), 1)
    cv2.circle(radar, (margin+pw//2, margin+ph//2), ph//6, (255,255,255), 1)
    
    # Project positions
    positions_frame = project_to_pitch(person_dets_frame, labels_frame, H, frame)
    for px, py, team in positions_frame:
        rx = int(margin + (px / PITCH_LENGTH) * pw)
        ry = int(margin + (py / PITCH_WIDTH) * ph)
        color = (255, 100, 100) if team == team_a_idx else (100, 100, 255) if team == team_b_idx else (0, 255, 255)
        cv2.circle(radar, (rx, ry), 8, color, -1)
        cv2.circle(radar, (rx, ry), 8, (255,255,255), 1)
    
    # Side by side
    combined = np.hstack([annotated_frame, radar])
    out.write(combined)
    frame_count += 1

cap.release()
out.release()
print(f'Processed {frame_count} frames → /content/tactical_output.mp4')

## Next Steps

- **Automated pitch keypoints:** Use Roboflow pitch keypoint model instead of manual points
- **Ball tracking:** Highlight ball position with different marker
- **Speed/distance:** Calculate from frame-to-frame position changes
- **Drone footage:** Nadir view simplifies homography (affine transform only)
- **Multi-sport:** Swap field template for basketball/hockey/tactical terrain
- **Real-time:** Process live RTSP stream with MQTT position output